###Importing required libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import uuid

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS quickcart.default.bronze


In [0]:
'''
spark.sql("""
CREATE SCHEMA IF NOT EXISTS quickcart.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS quickcart.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS quickcart.gold
""")
'''

In [0]:
spark.sql("SHOW SCHEMAS IN quickcart").show()

###Initializing path

In [0]:

CATALOG = "quickcart"
SOURCE_SCHEMA = "default"
BRONZE_SCHEMA = "bronze"

SOURCE_VOLUME = f"/Volumes/{CATALOG}/{SOURCE_SCHEMA}/source_data"
BRONZE_PATH = f"{CATALOG}.{BRONZE_SCHEMA}"



###Customer

In [0]:
CUSTOMER_SOURCE_PATH = f"{SOURCE_VOLUME}/customers"
CUSTOMER_BRONZE_CHECKPOINT = f"{SOURCE_VOLUME}/_checkpoints/customers"
CUSTOMER_BRONZE_SCHEMA = f"{SOURCE_VOLUME}/_schemas/customers"

CUSTOMER_BRONZE_PATH = f"{BRONZE_PATH}.customers"

In [0]:
customer_bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", CUSTOMER_BRONZE_SCHEMA)
    .load(CUSTOMER_SOURCE_PATH)
)

customer_bronze_stream = (
    customer_bronze_stream
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.expr("_metadata.file_path")
    )
)

customer_bronze_stream.printSchema()

In [0]:
customer_stream_data = customer_bronze_stream.writeStream.format("delta")\
    .outputMode("append").option("checkpointLocation",CUSTOMER_BRONZE_CHECKPOINT)\
        .trigger(availableNow = True).toTable(CUSTOMER_BRONZE_PATH)

customer_stream_data.awaitTermination()

In [0]:
%sql
SELECT *
FROM quickcart.bronze.customers
LIMIT 20;

In [0]:
%sql
SELECT *
FROM quickcart.bronze.customers
where _rescued_data is not null;